# 02 Feature Engineering

Build chronological, leakage-safe team features from prior matches only.

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

ARTIFACTS_DIR = Path("../artifacts")
clean_matches = pd.read_csv(ARTIFACTS_DIR / "clean_matches.csv", parse_dates=["date"])


def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))


def safe_mean(values: list[int], default: float = 0.5) -> float:
    return float(np.mean(values)) if values else default


def build_match_features_local(
    df: pd.DataFrame,
    base_elo: float = 1500.0,
    k_factor: float = 32.0,
) -> pd.DataFrame:
    ordered = df.sort_values(["date"], kind="mergesort").reset_index(drop=True).copy()
    elo = defaultdict(lambda: base_elo)
    matches_played = defaultdict(int)
    overall_history = defaultdict(list)
    blue_side_history = defaultdict(list)
    red_side_history = defaultdict(list)
    last_played = {}
    head_to_head = defaultdict(list)
    rows = []

    for row in ordered.itertuples(index=False):
        blue = row.blue_team
        red = row.red_team
        match_date = row.date
        key = tuple(sorted((blue, red)))
        blue_h2h = head_to_head[key]

        rows.append(
            {
                "season": row.season,
                "date": match_date,
                "event": row.event,
                "patch": row.patch,
                "blue_team": blue,
                "red_team": red,
                "winner": row.winner,
                "blue_team_win": row.blue_team_win,
                "elo_diff": elo[blue] - elo[red],
                "winrate_last_5_diff": safe_mean(overall_history[blue][-5:]) - safe_mean(overall_history[red][-5:]),
                "winrate_last_10_diff": safe_mean(overall_history[blue][-10:]) - safe_mean(overall_history[red][-10:]),
                "winrate_last_20_diff": safe_mean(overall_history[blue][-20:]) - safe_mean(overall_history[red][-20:]),
                "matches_played_diff": matches_played[blue] - matches_played[red],
                "days_since_last_match_diff": (
                    (match_date - last_played[blue]).days if blue in last_played else -1
                ) - (
                    (match_date - last_played[red]).days if red in last_played else -1
                ),
                "head_to_head_winrate_diff": (
                    sum(1 for winner in blue_h2h if winner == blue) / len(blue_h2h) if blue_h2h else 0.5
                ) - 0.5,
                "blue_side_team_winrate": safe_mean(blue_side_history[blue]),
                "red_side_team_winrate": safe_mean(red_side_history[red]),
            }
        )

        if pd.isna(row.blue_team_win):
            continue

        outcome = int(row.blue_team_win)
        expected = expected_score(elo[blue], elo[red])
        elo[blue] += k_factor * (outcome - expected)
        elo[red] += k_factor * ((1 - outcome) - (1 - expected))
        matches_played[blue] += 1
        matches_played[red] += 1
        overall_history[blue].append(outcome)
        overall_history[red].append(1 - outcome)
        blue_side_history[blue].append(outcome)
        red_side_history[red].append(1 - outcome)
        last_played[blue] = match_date
        last_played[red] = match_date
        head_to_head[key].append(blue if outcome == 1 else red)

    return pd.DataFrame(rows)


match_features = build_match_features_local(clean_matches)
match_features.to_csv(ARTIFACTS_DIR / "match_features.csv", index=False)
feature_df = match_features
feature_df.head()

In [2]:
feature_columns = [
    "elo_diff",
    "winrate_last_5_diff",
    "winrate_last_10_diff",
    "winrate_last_20_diff",
    "matches_played_diff",
    "days_since_last_match_diff",
    "head_to_head_winrate_diff",
    "blue_side_team_winrate",
    "red_side_team_winrate",
]
feature_df[feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
elo_diff,1070.0,1.912644,85.497166,-307.884347,-50.603533,0.000000,54.777720,298.167397
winrate_last_5_diff,1070.0,-0.003474,0.398615,-1.000000,-0.300000,0.000000,0.266667,1.000000
winrate_last_10_diff,1070.0,0.002978,0.332991,-1.000000,-0.200000,0.000000,0.200000,1.000000
winrate_last_20_diff,1070.0,0.001556,0.314649,-1.000000,-0.166667,0.000000,0.191106,1.000000
matches_played_diff,1070.0,0.229907,28.193894,-108.000000,-10.000000,0.000000,11.000000,107.000000
days_since_last_match_diff,1070.0,7.582243,172.818192,-1809.000000,0.000000,0.000000,0.000000,1813.000000
head_to_head_winrate_diff,1070.0,-0.002853,0.335982,-0.500000,-0.166667,0.000000,0.166667,0.500000
blue_side_team_winrate,1070.0,0.607587,0.251439,0.000000,0.500000,0.623311,0.750000,1.000000
red_side_team_winrate,1070.0,0.486518,0.270618,0.000000,0.333333,0.500000,0.641234,1.000000


In [3]:
feature_df[["date", "blue_team", "red_team", "blue_team_win", "elo_diff"]].head(10)

,date,blue_team,red_team,blue_team_win,elo_diff
0,2011-06-18,TSM,Team_gamed_21-de,1,0.0
1,2011-06-18,Counter_Logic_Gaming,Xan,1,0.0
2,2011-06-18,TSM,Counter_Logic_Gaming,1,0.0
3,2011-06-18,Team_gamed_21-de,Xan,1,0.0
4,2011-06-18,TSM,Xan,0,64.0
5,2011-06-18,Team_gamed_21-de,Counter_Logic_Gaming,0,0.0
6,2011-06-18,Against_All_authority,Pacific_eSports,1,0.0
7,2011-06-18,Epik_Gamer,Fnatic,1,0.0
8,2011-06-18,Against_All_authority,Fnatic,1,32.0
9,2011-06-18,Epik_Gamer,Pacific_eSports,1,32.0


In [4]:
feature_df.isna().mean().sort_values(ascending=False).head(20)

patch                         0.114019
date                          0.000000
red_team                      0.000000
blue_team                     0.000000
season                        0.000000
event                         0.000000
blue_team_win                 0.000000
elo_diff                      0.000000
winrate_last_5_diff           0.000000
winrate_last_10_diff          0.000000
winrate_last_20_diff          0.000000
matches_played_diff           0.000000
days_since_last_match_diff    0.000000
head_to_head_winrate_diff     0.000000
blue_side_team_winrate        0.000000
red_side_team_winrate         0.000000
dtype: float64